In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd

from pyhdas.frequency import spectrogram, add_db
from pyhdas.aggregate import quantile
from pyhdas.aragon import concat_raw_data, aragon_select_files

from datetime import timedelta

In [ ]:
import psutil

# Total memory
total_memory = psutil.virtual_memory().total / (1024 ** 3)  # in GB
# Available memory
available_memory = psutil.virtual_memory().available / (1024 ** 3)  # in GB
# Used memory
used_memory = psutil.virtual_memory().used / (1024 ** 3)  # in GB

print(f"Total memory: {total_memory:.2f} GB")
print(f"Available memory: {available_memory:.2f} GB")
print(f"Used memory: {used_memory:.2f} GB")

In [ ]:
dir_data = Path(r"data/19")
t_start = pd.Timestamp("2021-02-19 10:00:00+00")
t_end = pd.Timestamp("2021-02-19 10:04:00+00")
extension = "bin"

In [ ]:
file_list = list(aragon_select_files(dir_data, t_start, t_end, extension=extension))
ds_raw = concat_raw_data(file_list)

In [ ]:
t_start = t_start.tz_localize(None)
t_end = t_end.tz_localize(None)
ds_raw = ds_raw.sel(time=slice(t_start, t_end))

In [ ]:
# Plot strain
fig,ax = plt.subplots(figsize=(12,6))
for pos in np.arange(4220, 4260, 10):
    ax.plot(ds_raw.time, ds_raw.sel(position=pos).strain, label=f"{pos}m")
ax.set_ylabel("strain [m/m]")
ax.legend(loc="best", fancybox=True, shadow=True)
ax.set_title("strain timeseries at multiple locations")
plt.show()

In [ ]:
events = pd.read_csv("events_table.csv", parse_dates=["start", "end"])
events = events[(events["start"] >= t_start) & (events["end"] <= t_end)]

In [ ]:
def get_color(label):
    color_map = {
        "label_0": "gray",
        "label_1": "red",
        "label_2": "blue",
        "label_3": "green",
        "label_4": "yellow",
        "label_5": "orange",
        "label_6": "purple",
        "label_7": "magenta",
    }
    return color_map.get(label, "black")  

def plot_with_events(loc):

    loc_events = events[events['poi'].apply(lambda x: str(loc) in x)]
    fig, ax = plt.subplots(figsize=(12,6))
    ax.plot(ds_raw.time, ds_raw.sel(position=loc).strain, label=f"Strain at {loc}m")
    
    seen_labels = set()
    
    for _, event in loc_events.iterrows():
        start, end, label = event["start"], event["end"], event["label_anon"]
        color = get_color(label)

        if label not in seen_labels:
            ax.axvspan(start, end, alpha=0.3, label=f"{label}", color=color)
            seen_labels.add(label)  
        else:
            ax.axvspan(start, end, alpha=0.3, color=color) 
    
    
    ax.set_ylabel("strain [m/m]")
    ax.set_title(f"Strain timeseries at {loc}")
    ax.legend(loc="best", fancybox=True, shadow=True)
    plt.show()


for pos in np.arange(4220, 4260, 10):
    plot_with_events(pos)

In [ ]:
# Time window of 1 minute with 30 seconds overlap: was selected based on duration distribution
window_size = timedelta(minutes=1)
overlap = timedelta(seconds=30)
windows = []
current_start = t_start

while current_start + window_size <= t_end:
    current_end = current_start + window_size
    windows.append((current_start, current_end))
    current_start += overlap

results = []

for loc in ds_raw.position:

    # Calculate features for each time window for each location
    for start_time, end_time in windows:

        window_data = ds_raw.sel(time=slice(start_time, end_time), position=loc.item()).strain
        
        # Getting some basic stats about strain data
        mean_strain = window_data.mean().item()
        max_strain = window_data.max().item()
        range_strain = (window_data.max() - window_data.min()).item()
        mean_rate_of_change = window_data.diff('time').mean().item()
        max_rate_of_change = window_data.diff('time').max().item()
        
        window_labels = []
        for _, event in events.iterrows():
            
            event_pois = [int(poi.strip()) for poi in str(event["poi"]).split(",")]

            if loc in event_pois:

                event_start, event_end, label = event["start"], event["end"], event["label_anon"]
                
                # Overlap between window and event
                overlap_start = max(start_time, event_start)
                overlap_end = min(end_time, event_end)
                overlap_duration = (overlap_end - overlap_start).total_seconds()
                
                # Checking if overlap is at least half the duration of the event (can be changed, especially due to typical events durations)
                event_duration = (event_end - event_start).total_seconds()
                if overlap_duration >= event_duration / 2:
                    window_labels.append(label)
        
        results.append({
            "location": loc.item(),
            "start_time": start_time,
            "end_time": end_time,
            "mean_strain": mean_strain,
            "max_strain": max_strain,
            "range": range_strain,
            "mean_rate_of_change": mean_rate_of_change,
            "max_rate_of_change": max_rate_of_change,
            "labels": ', '.join(window_labels) if window_labels else "Normal"
        })

    df_features = pd.DataFrame(results)


# Frequency analysis

In [ ]:
ds_spect = spectrogram(ds_raw, variable='strain')

In [ ]:
# Plot spectrogram at a specific location
fig,ax = plt.subplots(figsize=(15,6))
ds_spect.Pxx_dB.sel(position=4240).plot(x="time", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
ax.set_title("spectrogram for location 4240m")
plt.show()

In [ ]:
ds_spect.freq

In [ ]:
# Same time window duration
window_size = timedelta(minutes=1)
overlap = timedelta(seconds=30)
windows = []
current_start = t_start

while current_start + window_size <= t_end:
    current_end = current_start + window_size
    windows.append((current_start, current_end))
    current_start += overlap

features_with_labels = []

for pos in ds_spect.position:
    
    position_events = events[events["poi"].str.contains(str(pos.item()))]
    
    for start_time, end_time in windows:

        data = ds_spect.sel(position=pos, time=slice(start_time, end_time))
        
        total_power = data.Pxx_dB.sum().item()
        low_band_power = data.Pxx_dB.sel(freq=slice(0, 100)).sum().item()
        mid_band_power = data.Pxx_dB.sel(freq=slice(100, 500)).sum().item()
        high_band_power = data.Pxx_dB.sel(freq=slice(500, 1000)).sum().item()
        #peak_freq = data.Pxx.argmax("freq").item()
        mean_power = data.Pxx_dB.mean().item()
        std_power = data.Pxx_dB.std().item()
        p_norm = data.Pxx_dB / data.Pxx_dB.sum()
        spectral_entropy = -np.nansum(p_norm * np.log2(p_norm)).item()
        
        window_labels = []
        for _, event in events.iterrows():
        
            event_pois = [int(poi.strip()) for poi in str(event["poi"]).split(",")]

            if pos in event_pois:

                event_start, event_end, label = event["start"], event["end"], event["label_anon"]
                
                overlap_start = max(start_time, event_start)
                overlap_end = min(end_time, event_end)
                overlap_duration = (overlap_end - overlap_start).total_seconds()
                
                event_duration = (event_end - event_start).total_seconds()
                if overlap_duration >= event_duration / 2:
                    window_labels.append(label)
        
        # Store results
        features_with_labels.append({
            "position": pos.item(),
            "start_time": start_time,
            "end_time": end_time,
            "total_power": total_power,
            "low_band_power": low_band_power,
            "mid_band_power": mid_band_power,
            "high_band_power": high_band_power,
            #"peak_freq": peak_freq,
            "mean_power": mean_power,
            "std_power": std_power,
            "spectral_entropy": spectral_entropy,
            "labels": ', '.join(window_labels) if window_labels else "Normal"
        })

df_features_with_labels = pd.DataFrame(features_with_labels)

In [ ]:

df = df_features_with_labels  

df['start_time'] = pd.to_datetime(df['start_time'])
df["label_type"] = df["labels"].apply(lambda x: "Anomaly" if x != "Normal" else "Normal")

color_map = {"Normal": "skyblue", "Anomaly": "orange"}

metrics = ["low_band_power", "mid_band_power", "high_band_power", 
           "mean_power", "std_power", "spectral_entropy"]

for start_time in df["start_time"].unique():

    window_data = df[(df["start_time"] == start_time) & (df["position"] < 6900)]

    for metric in metrics:
        plt.figure(figsize=(12, 6))
        
        for idx, row in window_data.iterrows():
            color = color_map[row["label_type"]]
            plt.bar(row["position"], row[metric], color=color, alpha=0.7, width=20)  # Increased bar width for visibility
        
        plt.title(f"{metric} across Locations for Time Window {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        plt.xlabel("Location")
        plt.ylabel(metric)
        
        handles = [plt.Line2D([0], [0], color=color_map["Normal"], lw=4),
                   plt.Line2D([0], [0], color=color_map["Anomaly"], lw=4)]
        plt.legend(handles, ["Normal", "Anomaly"])
        
        plt.show()


In [ ]:
df_features_with_labels.to_csv("spectra_features.csv", index=False)

In [ ]:
# Plot frequency spectrum at a specific location and timestamp
fig,ax = plt.subplots(figsize=(12,6))
da = ds_spect.sel(position=4240, time="2021-02-25 10:01:32")
ax.plot(da.freq, da.Pxx_dB)
ax.set_xlabel("frequency [Hz]")
ax.set_ylabel("Pxx_dB [dB]")
ax.set_title(f"frequency spectrum at position 4240m at {da.time.values[0]}")
plt.show()

In [ ]:
ds_spect

In [ ]:
# Plot frequency spectra for all positions at specific timestamp
fig,ax = plt.subplots(figsize=(15,6))
ds_spect.Pxx_dB.isel(time=0).plot(x="position", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma")
ax.set_title(f"frequency spectra per position at {ds_spect.time.values[0]}")
plt.show()

In [ ]:
# Aggregate spectogram over time (e.g. pick the 10% quantile over time -> filters out sudden loud noises) 
ds_spect_quantile = ds_spect.reduce(func=lambda x, axis: np.nanquantile(x, q=0.9, axis=axis), dim="time")

In [ ]:
fig,ax = plt.subplots(figsize=(15,6))
ds_spect_quantile.Pxx_dB.plot(x="position", y="freq", ax=ax, vmin=-10, vmax=20, cmap="magma")
ax.set_title(f"frequency spectra aggregated (10% quantile) over 5min per position[dB]")
plt.show()

# Soundlevel analysis

In [ ]:
ds_soundlevel = ds_spect[["Pxx"]].sum(dim="freq")
ds_soundlevel = add_db(ds_soundlevel)

ds_soundlevel_low_pass = ds_spect[["Pxx"]].sel(freq=slice(0,100)).sum(dim="freq")
ds_soundlevel_low_pass = add_db(ds_soundlevel_low_pass)

ds_soundlevel_mid_pass = ds_spect[["Pxx"]].sel(freq=slice(100,500)).sum(dim="freq")
ds_soundlevel_mid_pass = add_db(ds_soundlevel_mid_pass)

ds_soundlevel_high_pass = ds_spect[["Pxx"]].sel(freq=slice(500,1000)).sum(dim="freq")
ds_soundlevel_high_pass = add_db(ds_soundlevel_high_pass)

In [ ]:
# Plot soundlevel over time and all positions
fig,ax = plt.subplots(figsize=(15,6))
ds_soundlevel.Pxx_dB.plot(x="position", y="time", ax=ax, vmax = 40, cmap="magma")
ax.set_title(f"soundlevel [dB] over all frequencies")
plt.show()

In [ ]:
# Plot soundlevel timeseries at a selection of locations
fig,ax = plt.subplots(figsize=(12,6))
for pos in np.arange(4220, 4270, 10):
    ax.plot(ds_soundlevel.time, ds_soundlevel.sel(position=pos).Pxx_dB, label=f"{pos}m", alpha=0.5)
ax.legend(loc="best", fancybox=True, shadow=True)
ax.set_ylabel("Pxx_dB [dB]")
ax.set_title(f"soundlevel timeseries at multiple locations")
plt.show()

In [ ]:
# Plot high passed soundlevel over time and all positions -> filters out cars
fig,ax = plt.subplots(figsize=(15,6))
ds_soundlevel_high_pass.Pxx_dB.plot(x="position", y="time", ax=ax, vmin = 0, vmax = 40, cmap="magma")
ax.set_title(f"high pass soundlevel [dB]")
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(15,6))
ds_soundlevel_mid_pass.Pxx_dB.plot(x="position", y="time", ax=ax, vmin = 0, vmax = 40, cmap="magma")
ax.set_title(f"mid pass soundlevel [dB]")
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(15,6))
ds_soundlevel_low_pass.Pxx_dB.plot(x="position", y="time", ax=ax, vmin = 0, vmax = 40, cmap="magma")
ax.set_title(f"low pass soundlevel [dB]")
plt.show()

# hvplot examples

In [ ]:
import hvplot.xarray

In [ ]:
ds_soundlevel.sel(position=slice(4220,4260)).hvplot(
    x="time", y="Pxx_dB", by="position", width=1200, height=600, title="soundlevel timeseries at multiple locations"
) # By clicking in the legend you can make positions transparent

In [ ]:
ds_soundlevel.Pxx_dB.hvplot(
    x="position", y="time", width=1200, height=600, clim=(None,40), cmap="magma", title="soundlevel [dB] over all frequencies"
)